# aiswakepy â€” Run Pipeline

Edit the paths and parameters in **Cell 1** before running.  
Each cell is independent â€” re-run any stage without restarting the kernel.

In [ ]:
# === EDIT THESE PATHS FOR YOUR RUN ===
work_directory   = r"C:/Projects/61802721 Shoreline Stabilisation at Tuas Curve Profile (sub)/MATLAB"   # root folder for this run; change to switch datasets

ais_csv          = f"{work_directory}/1_Data/AIS/2023/DTshipwake_test.csv"
coastline_shp    = f"{work_directory}/1_Data/AIS/2023/Shoreline_M21SW_CD1_baseline.shp"
bathy_file       = f"{work_directory}/1_Data/AIS/2023/TCP_HD_Prod_mCD_LongLat_6mV08_CD1_Opt3_Baseline.mesh"
tide_dfs0        = f"{work_directory}/1_Data/AIS/2023/TCP_AIS_WL.dfs0"
tide_item        = "NBS: Surface elevation"   # item name inside the .dfs0 file
study_area_shp   = None   # optional polygon shapefile to clip study area, or None

# === OUTPUT FILENAMES (change to save different runs side by side) ===
output_dir           = f"{work_directory}/4_Final/AISWakePy/output"
wave_height_map_name = "WaveHeightMap.png"
wave_period_map_name = "WavePeriodMap.png"
wave_params_name     = "wave_params.csv"
shore_impact_name    = "shore_impact.csv"
save_stage_csv       = True   # save {ais_stem}_filtered/depth/wave/impact.csv as checkpoints

# === PLOT VISUALIZATION PARAMETERS ===
plot_show        = True   # show plots inline in notebook
plot_lon0        = None   # plot longitude range (None = auto-fit to data)
plot_lon1        = None
plot_lat0        = None   # plot latitude range (None = auto-fit to data)
plot_lat1        = None
plot_zoom        = "auto"  # tile zoom level ("auto" = contextily calculates from extent, or specify integer 0-18)

# === DATA FILTER RANGE (crops the final shore-impact results and all downstream plots/tables) ===
filter_lon0      = None   # crop longitude range (None = no crop)
filter_lon1      = None
filter_lat0      = None   # crop latitude range (None = no crop)
filter_lat1      = None

# === AIS FILTER PARAMETERS ===
traj_gap_s           = 180.0   # split trajectory on gap > this (seconds)
max_velocity_knots   = 36.0    # kinematic check: flag segment if avg speed exceeds this
max_acceleration_ms2 = 20      # acceleration check: replace SOG/COG if implied accel exceeds this
interp_interval_s    = 30.0    # Hermite spline interpolation time step (seconds)

# === WAVE CALCULATION PARAMETERS ===
formula        = "kriebel"  # empirical wake model: 'bhowmik', 'blaauw', 'gates', 'kriebel', 'maynord', 'pianc', 'sorensen'
cb_method      = "B_Le"     # block coefficient method: 'L_Le' (Kriebel default), 'B_Le', or 'table'
                            #   L_Le: Cb estimated from waterline length / bow entry length ratio
                            #   B_Le: uses beam instead of length; table: lookup by vessel type
gravity        = 9.78       # local gravitational acceleration (m/s²); Singapore value (vs 9.81 standard)
max_prop_m     = 2000.0     # maximum wake ray propagation distance from vessel track (m);
                            #   rays beyond this are discarded before coastline intersection
wake_cutoff_m  = 0.01       # minimum shore wave height to record (m); events below this are dropped

# === GENERAL WAKE FILTER PARAMETERS (applied regardless of formula) ===
max_sog_knots  = 24.0   # maximum vessel speed (knots)        (fast craft outside calculation range)
max_bl_ratio   = 1.0    # maximum beam / length ratio         (atypical hull form, model not valid)

# === KRIEBEL (2005) APPLICABILITY LIMITS (formula-specific; ignored for other formulas) ===
min_Froude_M   = 0.1    # minimum modified Froude number Fr*  (vessel must be in wave-making regime)
max_Froude_M   = 0.5    # maximum modified Froude number Fr*  (avoids supercritical planing regime)
max_bf         = 0.4    # maximum BF = Beta * (Fr*-0.1)^2     (no data in Kriebel's dataset exceeds 0.4)


In [ ]:
# === Cell 2: Build config ===
from aiswakepy.config import load_config

config = load_config({
    "ais": {
        "raw_csv": ais_csv,
        "traj_gap_s": traj_gap_s,
        "max_velocity_knots": max_velocity_knots,
        "max_acceleration_ms2": max_acceleration_ms2,
        "interp_interval_s": interp_interval_s,
        "study_area_shp": study_area_shp,
    },
    "vessel": {"cb_method": cb_method},
    "bathymetry": {
        "source": bathy_file,
        "tide_dfs0": tide_dfs0,
        "tide_item": tide_item,
    },
    "coastline": {"shapefile": coastline_shp},
    "wave": {
        "gravity": gravity,
        "min_Froude_M": min_Froude_M,
        "max_Froude_M": max_Froude_M,
        "max_bf": max_bf,
        "max_sog_knots": max_sog_knots,
        "max_bl_ratio": max_bl_ratio,
    },
    "impact": {"max_propagation_m": max_prop_m, "wake_cutoff_m": wake_cutoff_m},
    "output": {
        "directory": output_dir,
        "wave_height_map_name": wave_height_map_name,
        "wave_period_map_name": wave_period_map_name,
        "wave_params_name": wave_params_name,
        "shore_impact_name": shore_impact_name,
        "save_stage_csv": save_stage_csv,
    },
})
print("Config loaded OK")

In [ ]:
# === Cell 3: Stage 1 â€” AIS Filter ===
from pathlib import Path
from aiswakepy.stages.filter import filter_ais

df_filtered = filter_ais(
    csv_path=config.ais.raw_csv,
    coastline_shp=config.coastline.shapefile,
    gap_s=config.ais.traj_gap_s,
    max_velocity_knots=config.ais.max_velocity_knots,
    max_acceleration_ms2=config.ais.max_acceleration_ms2,
    interval_s=config.ais.interp_interval_s,
    study_area_shp=config.ais.study_area_shp,
)
print(f"Filtered and interpolated: {len(df_filtered):,} rows")
if config.output.save_stage_csv:
    _stem = Path(config.ais.raw_csv).stem
    _out = Path(config.output.directory)
    _out.mkdir(parents=True, exist_ok=True)
    df_filtered.to_csv(_out / f"{_stem}_01_filtered.csv", index=False)
    print(f"  saved {_stem}_01_filtered.csv")
df_filtered.head()

In [ ]:
# === Cell 4: Stage 2 â€” Depth + Tide Assignment ===
from aiswakepy.stages.depth import assign_depth

df_depth = assign_depth(
    df=df_filtered,
    bathy_path=config.bathymetry.source,
    tide_dfs0_path=config.bathymetry.tide_dfs0,
    tide_item=config.bathymetry.tide_item,
    underkeel_margin_m=config.bathymetry.underkeel_margin_m,
)
print(f"After depth filter: {len(df_depth)} rows")
if config.output.save_stage_csv:
    df_depth.to_csv(_out / f"{_stem}_02_depth.csv", index=False)
    print(f"  saved {_stem}_02_depth.csv")
df_depth[["WaterDepth", "draught"]].describe()

In [ ]:
# === Cell 5: Stage 3 â€” Vessel Parameters ===
from aiswakepy.stages.vessel import compute_vessel_params

print(
    f"Vessel parameters  cb_method: {cb_method}\n"
    f"  general filters  SOG <= {max_sog_knots} kn  B/L <= {max_bl_ratio}"
)

df_vessel = compute_vessel_params(
    df=df_depth,
    cb_method=config.vessel.cb_method,
    g=config.wave.gravity,
    max_sog_knots=config.wave.max_sog_knots,
    max_bl_ratio=config.wave.max_bl_ratio,
)
print(f"Vessel events: {len(df_vessel)}")
if config.output.save_stage_csv:
    df_vessel.to_csv(_out / f"{_stem}_03_vessel.csv", index=False)
    print(f"  saved {_stem}_03_vessel.csv")
df_vessel[["SOGms", "Froude_D", "T", "Theta", "Cel", "Tc", "BLratio"]].describe()

In [ ]:
# === Cell 6: Stage 4 â€” Wave Impact ===
from aiswakepy.stages.wave_impact import compute_wave_impact

print(
    f"Wave impact  formula: {formula}\n"
    f"  Kriebel limits  Froude_M = [{min_Froude_M}, {max_Froude_M}]  BF <= {max_bf}"
)

df_wave_impact = compute_wave_impact(
    df_vessel=df_vessel,
    coastline_shp=config.coastline.shapefile,
    formula=formula,
    max_propagation_m=config.impact.max_propagation_m,
    wake_cutoff_m=config.impact.wake_cutoff_m,
    g=config.wave.gravity,
    rho=config.wave.rho_water,
    # Kriebel-specific applicability limits (ignored for other formulas)
    min_Froude_M=config.wave.min_Froude_M,
    max_Froude_M=config.wave.max_Froude_M,
    max_bf=config.wave.max_bf,
)
print(f"Shore impact events: {len(df_wave_impact)}")
if config.output.save_stage_csv:
    df_wave_impact.to_csv(_out / f"{_stem}_04_wave_impact.csv", index=False)
    print(f"  saved {_stem}_04_wave_impact.csv")
df_wave_impact.head()

In [ ]:
# === Cell 6b: Crop shore-impact results by lon/lat range ===
# Filters df_wave_impact in place so all downstream plots/tables/exports use the cropped set.
# Full results remain saved in the stage CSV (_04_wave_impact.csv).
_before = len(df_wave_impact)
if filter_lon0 is not None:
    df_wave_impact = df_wave_impact[df_wave_impact["ShLongitude"] >= filter_lon0]
if filter_lon1 is not None:
    df_wave_impact = df_wave_impact[df_wave_impact["ShLongitude"] <= filter_lon1]
if filter_lat0 is not None:
    df_wave_impact = df_wave_impact[df_wave_impact["ShLatitude"] >= filter_lat0]
if filter_lat1 is not None:
    df_wave_impact = df_wave_impact[df_wave_impact["ShLatitude"] <= filter_lat1]
df_wave_impact = df_wave_impact.reset_index(drop=True)
print(f"Cropped: {_before:,} -> {len(df_wave_impact):,} rows")


In [ ]:
# === Cell 7: Visualisation ===
from pathlib import Path
from aiswakepy.viz.wave_map import plot_wave_height_map, plot_wave_period_map

out = Path(output_dir)
out.mkdir(parents=True, exist_ok=True)

plot_wave_height_map(
    df_wave_impact, coastline_shp, out / wave_height_map_name,
    max_points=100_000,
    lon0=plot_lon0, lon1=plot_lon1, lat0=plot_lat0, lat1=plot_lat1,
    zoom=plot_zoom,
    show=plot_show,
)
plot_wave_period_map(
    df_wave_impact, coastline_shp, out / wave_period_map_name,
    max_points=100_000,
    lon0=plot_lon0, lon1=plot_lon1, lat0=plot_lat0, lat1=plot_lat1,
    zoom=plot_zoom,
    show=plot_show,
)
print(f"Maps saved to {out.resolve()}")

In [ ]:
# === Cell 7b: Wave Height Histogram ===
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_wave_impact["WaveHeight"], bins=50, color="steelblue", edgecolor="black")
ax.set_xlabel("Wave Height (m)")
ax.set_ylabel("Count")
ax.set_title("Shore Wave Height Distribution")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(out / "WaveHeightHistogram.png", dpi=150, bbox_inches="tight")
if plot_show:
    plt.show()
plt.close(fig)
print(f"Histogram saved to {out / 'WaveHeightHistogram.png'}")


In [ ]:
# === Cell 7c: Top 10 Vessels by Peak Wave Height ===
top_vessels = (
    df_wave_impact.groupby("MMSI")["WaveHeight"]
    .agg(["max", "mean", "count"])
    .sort_values("max", ascending=False)
    .head(10)
    .rename(columns={
        "max":   "MaxWaveHeight_m",
        "mean":  "MeanWaveHeight_m",
        "count": "ImpactEvents",
    })
)
top_vessels.to_csv(out / "top10_vessels.csv")
top_vessels


In [ ]:
# === Cell 8: Export final results ===
# Stage CSVs are already saved above.
# This cell saves final outputs using the configured filenames.
from pathlib import Path

out = Path(output_dir)
df_vessel.to_csv(out / wave_params_name, index=False)
df_wave_impact.to_csv(out / shore_impact_name, index=False)
print(f"Results saved to {out.resolve()}")